# 09. Statistical Evaluation: TRTR vs TSTR / TSRTR Benchmark

This notebook provides a rigorous statistical comparison between two training regimens,
evaluated on the same held-out CV validation spectra:
- **TRTR**: Train on Real, Test on Real (baseline)
- **TSTR / TSRTR**: synthetic-augmented regimen (candidate, selected via `REGIMEN`)

## Statistical design

- **Slide-level clustering**: the cluster bootstrap resamples *slides* (full biopsy-scan
  IDs, e.g. `HF-2102_V3AM_2.h5_5`), not individual spectra. Within-slide spectral
  correlation is strong and across-slide weak, so the slide is the unit of resampling.
- **Honest predictions / no leakage**: each slide is scored only by the model from the
  fold where that slide was held out (fold-i models never predict fold-i slides).
- **Cached predictions**: predictions are computed once and reused across bootstrap
  iterations.
- **Primary inference**: the 95% percentile CI of the slide-level balanced-accuracy
  difference (candidate − baseline). A difference is flagged when its 95% CI excludes 0.
- **Secondary descriptor**: a two-sided bootstrap p-value (`p_bootstrap`), reported
  without multiple-comparison correction (see limitation).
- **Descriptive metrics** (spectrum level, pooled across folds): accuracy, balanced
  accuracy, F1 (macro/weighted), precision, recall, Cohen's κ, and — for binary tasks —
  sensitivity, specificity, AUROC, AUPRC. These populate the paper results tables.

All statistics are computed by `src/utils/statistical_evaluation.py` — the same module the
batch pipeline (`pipeline/evaluation_stage.py`) uses — so the two paths produce identical
numbers.

**Limitation**: no correction is applied for multiple comparisons across models or across
the experiment suite.

## 1. Imports & setup

In [ ]:
import os
import sys
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# Project root on path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebooks.experiment_utils import setup_experiment
from pipeline._common import make_loader, num_classes_of
from pipeline._cvae_gen import build_cvae_config

# Single source of truth for all statistics (shared with pipeline/evaluation_stage.py)
from src.utils.statistical_evaluation import (
    find_model_path,
    load_and_predict,
    load_and_predict_proba,
    stratified_slide_accuracy,
    stratified_slide_bootstrap,
    compute_full_metrics,
    summarize_bootstrap_diffs,
    build_summary_rows,
    compute_cbr_predictions_per_fold,
    stratified_slide_bootstrap_cbr,
)

print("Imports successful")

## 2. Experiment configuration

In [ ]:
# ── Experiment + regimen selection ─────────────────────────────────────
CONFIG_NAME = 'exp_104'     # any configs/exp_*.yaml with trained TRTR + candidate + CVAE models
REGIMEN     = 'TSRTR'       # candidate to compare against the TRTR baseline: 'TSTR' or 'TSRTR'

config, output_dir = setup_experiment(CONFIG_NAME)

candidate_dir = (config['paths']['models_tsrtr_dir'] if REGIMEN == 'TSRTR'
                 else config['paths']['models_tstr_dir'])

CFG = {
    "model_a_dir": config['paths']['models_trtr_dir'],   # TRTR baseline
    "model_b_dir": candidate_dir,                         # candidate (TSTR or TSRTR)
    "cvae_dir":    config['paths']['models_cvae_dir'],    # for Classification by Reconstruction
    "output_dir":  config['paths']['statistical_dir'],
    "n_bootstrap": config['statistics']['bootstrap_samples'],
    "alpha":       config['statistics']['alpha'],
    "random_seed": config['statistics']['random_seed'],
    "n_folds":     config['cv_strategy']['n_folds'],
    "primary_metric": config['statistics']['primary_metric'],
    "models": {
        "Apollo":      {"prefix": "apollo", "ext": ".joblib", "type": "sklearn"},
        "RF_Standard": {"prefix": "rf",     "ext": ".joblib", "type": "sklearn"},
        "XGBoost":     {"prefix": "xgb",    "ext": ".json",   "type": "xgboost"},
        "1D_CNN":      {"prefix": "cnn",    "ext": ".keras",  "type": "keras"},
    },
}

np.random.seed(CFG["random_seed"])
os.makedirs(CFG["output_dir"], exist_ok=True)

print(f"Experiment : {CONFIG_NAME}  (baseline TRTR vs candidate {REGIMEN})")
print(f"Bootstrap  : {CFG['n_bootstrap']} iters | alpha={CFG['alpha']} | folds={CFG['n_folds']}")
print(f"Baseline   : {CFG['model_a_dir']}")
print(f"Candidate  : {CFG['model_b_dir']}")

## 3. Validation data & slide-to-fold mapping

In [ ]:
# Load the full (cropped) dataset once; the loader handles preprocessing/cropping.
print("Loading data...")
loader = make_loader(config)
actual_n_features = loader.X.shape[1]
num_classes = num_classes_of(config)
print(f"Input shape: ({actual_n_features}, 1) | num_classes={num_classes}")

# CVAE architecture for Classification by Reconstruction — the same helper the CVAE
# stage and evaluation_stage use (topology identical; gamma/delta unused at inference).
CVAE_CONFIG = build_cvae_config(config, actual_n_features)

# ── Per-fold validation spectra + per-spectrum slide IDs (no leakage) ──────────
# Each fold's validation slides are scored only by that fold's models. The slide ID
# is the FULL group string (one biopsy scan) — the cluster unit for the bootstrap.
# Selecting val rows via np.where(isin(...)) preserves the loader's ascending
# global-array order, so X_val / y_val / slide_ids stay row-aligned here AND align
# with the CbR predictions in the next cell (same loader, same fold order).
split_file = config['paths']['splits_file']
with open(split_file, 'r') as f:
    splits_data = json.load(f)
splits = splits_data.get('cv_folds', splits_data)

all_groups = np.array(loader.groups)
loader_iter = iter(loader)
X_val_list, y_val_list, slide_ids_list = [], [], []

for fold_idx in range(CFG['n_folds']):
    fold_data = next(loader_iter)
    fold_name = fold_data['fold_name']

    X_val = fold_data['val'][0].astype(np.float32)
    y_val = fold_data['val'][1].copy()
    if np.min(y_val) == 1:        # 1-indexed -> 0-indexed
        y_val = y_val - 1
    if num_classes == 2:          # binary task mapping (matches notebook 02)
        y_val = np.where(y_val < 3, 0, 1)

    raw_string_ids = splits[fold_name]['test']
    val_indices_sorted = np.where(np.isin(all_groups, raw_string_ids))[0]
    fold_slide_ids = all_groups[val_indices_sorted]

    X_val_list.append(X_val)
    y_val_list.append(y_val)
    slide_ids_list.append(fold_slide_ids)
    print(f"  {fold_name}: {len(y_val)} spectra over {len(np.unique(fold_slide_ids))} slides")

y_all = np.concatenate(y_val_list)
slide_ids_all = np.concatenate(slide_ids_list)
print(f"\nUnified test set: {len(y_all)} spectra over {len(np.unique(slide_ids_all))} slides "
      f"(classes {np.unique(y_all)})")

## 4. Classification-by-Reconstruction predictions

In [ ]:
# CbR predictions, pooled across folds in the SAME loader order as y_all /
# slide_ids_all above, so they stay row-aligned. Only the predictions are used; the
# scoring pairs them with the binary-mapped y_all + slide_ids_all master arrays.
# (Assumes every fold has CVAE weights; a missing fold would be skipped and misalign.)
cbr_preds, _, _ = compute_cbr_predictions_per_fold(
    loader=loader,
    cvae_config=CVAE_CONFIG,
    model_dir=CFG['cvae_dir'],
    n_folds=CFG['n_folds'],
)

## 5. Statistical evaluation (slide-level cluster bootstrap)

In [ ]:
# ---------------------------------------------------------
# SECTION 5: Statistical Evaluation (slide-level cluster bootstrap, no leakage)
#
# Primary inference: 95% percentile CI of the slide-level balanced-accuracy
# difference (candidate - baseline). Raw bootstrap p reported as a secondary
# descriptor; no multiplicity correction (disclosed as a limitation).
# Every statistic comes from src/utils/statistical_evaluation.py, so this
# notebook and pipeline/evaluation_stage.py produce identical numbers.
# ---------------------------------------------------------
results_all = []
n_folds = CFG['n_folds']

print(f"\n{'='*70}")
print(f"STATISTICAL EVALUATION: TRTR vs {REGIMEN} (Slide-Level Bootstrap)")
print(f"{'='*70}")

for model_name, m_cfg in CFG['models'].items():
    print(f"\n{'-'*70}\nMODEL: {model_name}\n{'-'*70}")

    preds_A_list, preds_B_list = [], []
    proba_A_list, proba_B_list = [], []

    for fold_idx in range(n_folds):
        X_val = X_val_list[fold_idx]
        try:
            path_A = find_model_path(CFG['model_a_dir'], m_cfg['prefix'], m_cfg['ext'], fold_idx)
            preds_A_list.append(load_and_predict(path_A, m_cfg['type'], X_val))
            path_B = find_model_path(CFG['model_b_dir'], m_cfg['prefix'], m_cfg['ext'], fold_idx)
            preds_B_list.append(load_and_predict(path_B, m_cfg['type'], X_val))
        except FileNotFoundError as e:
            print(f"  Missing model for {model_name} fold {fold_idx}: {e}")
            continue
        try:
            proba_A_list.append(load_and_predict_proba(path_A, m_cfg['type'], X_val))
            proba_B_list.append(load_and_predict_proba(path_B, m_cfg['type'], X_val))
        except Exception:
            pass

    if len(preds_A_list) < n_folds or len(preds_B_list) < n_folds:
        print(f"  {model_name}: Incomplete predictions. Skipping.")
        continue

    preds_A = np.concatenate(preds_A_list)
    preds_B = np.concatenate(preds_B_list)
    proba_A = np.concatenate(proba_A_list) if len(proba_A_list) == n_folds else None
    proba_B = np.concatenate(proba_B_list) if len(proba_B_list) == n_folds else None

    # Point estimate: slide-level balanced accuracy (the same statistic the bootstrap uses)
    slide_acc_A = stratified_slide_accuracy(preds_A, y_all, slide_ids_all)
    slide_acc_B = stratified_slide_accuracy(preds_B, y_all, slide_ids_all)

    # Descriptive metrics, all slide-level (consistent with the bootstrap):
    # decomposable metrics use the per-(class, slide) stratified mean, the rest
    # are per-slide values averaged over slides.
    full_A = compute_full_metrics(y_all, preds_A, proba_A, slide_ids=slide_ids_all)
    full_B = compute_full_metrics(y_all, preds_B, proba_B, slide_ids=slide_ids_all)
    print(f"  TRTR     - slide-level Acc={full_A['accuracy']:.4f}  F1={full_A['f1_macro']:.4f}  "
          f"Prec={full_A['precision_macro']:.4f}  Rec={full_A['recall_macro']:.4f}  kappa={full_A['kappa']:.4f}")
    print(f"  {REGIMEN} - slide-level Acc={full_B['accuracy']:.4f}  F1={full_B['f1_macro']:.4f}  "
          f"Prec={full_B['precision_macro']:.4f}  Rec={full_B['recall_macro']:.4f}  kappa={full_B['kappa']:.4f}")

    # Slide-level paired cluster bootstrap on the accuracy difference
    print(f"\n  Running slide-level bootstrap ({CFG['n_bootstrap']} iterations)...")
    _, _, boot_diffs = stratified_slide_bootstrap(
        preds_A, preds_B, y_all, slide_ids_all,
        n_boot=CFG['n_bootstrap'], seed=CFG['random_seed'])
    boot = summarize_bootstrap_diffs(boot_diffs)

    print(f"  Slide acc:  TRTR={slide_acc_A:.4f}  {REGIMEN}={slide_acc_B:.4f}  "
          f"delta={slide_acc_B - slide_acc_A:+.4f}")
    print(f"  Bootstrap delta ({REGIMEN} - TRTR): mean={boot['boot_mean_diff']:+.4f}  "
          f"95% CI[{boot['ci_diff_lo']:+.4f}, {boot['ci_diff_hi']:+.4f}]  "
          f"p={boot['p_bootstrap']:.4f}  CI-excludes-0={boot['significant_95']}")

    results_all.append({
        'model': model_name,
        'slide_acc_TRTR': slide_acc_A,
        f'slide_acc_{REGIMEN}': slide_acc_B,
        'delta': slide_acc_B - slide_acc_A,
        **boot,
        'full_metrics_TRTR': full_A,
        f'full_metrics_{REGIMEN}': full_B,
    })

# ============================================================
# CLASSIFICATION BY RECONSTRUCTION (reference metric; not a TRTR/candidate test)
# ============================================================
print(f"\n{'='*70}")
print("CLASSIFICATION BY RECONSTRUCTION: Bootstrap Confidence Interval")
print(f"{'='*70}\n")

boot_acc_cbr = stratified_slide_bootstrap_cbr(
    cbr_preds, y_all, slide_ids_all,
    n_boot=CFG['n_bootstrap'], seed=CFG['random_seed'])

cbr_acc_baseline = float(stratified_slide_accuracy(cbr_preds, y_all, slide_ids_all))
cbr_boot_mean    = float(np.mean(boot_acc_cbr))
cbr_ci_lo        = float(np.percentile(boot_acc_cbr, 2.5))
cbr_ci_hi        = float(np.percentile(boot_acc_cbr, 97.5))
# CbR is just another classifier: report its full slide-level metric set too.
cbr_full         = compute_full_metrics(y_all, cbr_preds, slide_ids=slide_ids_all)

print(f"  CbR slide accuracy: {cbr_acc_baseline:.4f}")
print(f"  Bootstrap mean:     {cbr_boot_mean:.4f}")
print(f"  95% CI: [{cbr_ci_lo:.4f}, {cbr_ci_hi:.4f}]")
print(f"  Slide-level F1={cbr_full['f1_macro']:.4f}  "
      f"prec={cbr_full['precision_macro']:.4f}  rec={cbr_full['recall_macro']:.4f}")

cbr_result = {
    'model': 'Classification_by_Reconstruction',
    'cbr_accuracy': cbr_acc_baseline,
    'cbr_boot_mean': cbr_boot_mean,
    'cbr_ci_lo': cbr_ci_lo,
    'cbr_ci_hi': cbr_ci_hi,
    'full_metrics': cbr_full,
    'n_slides': int(len(np.unique(slide_ids_all))),
    'n_samples': int(len(cbr_preds)),
    'n_iterations': CFG['n_bootstrap'],
    'seed': CFG['random_seed'],
}
# Flat metric keys read by paper/gen_appendix_tables.py (CbR row).
for _k in ('f1_macro', 'precision_macro', 'recall_macro', 'balanced_accuracy', 'kappa'):
    if _k in cbr_full:
        cbr_result[_k] = cbr_full[_k]
results_all.append(cbr_result)


## 6. Save & summarize results

In [ ]:
# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = Path(CFG['output_dir']) / f"statistical_evaluation_{timestamp}.json"
with open(json_path, 'w') as f:
    json.dump(results_all, f, indent=2)
print(f"Results saved: {json_path}")

# Flat CSV for paper tables — identical schema to pipeline/evaluation_stage.py
summary_df = pd.DataFrame(build_summary_rows(results_all, 'TRTR', REGIMEN))
csv_path = Path(CFG['output_dir']) / f"statistical_summary_{timestamp}.csv"
summary_df.to_csv(csv_path, index=False)
print(f"Summary CSV saved: {csv_path}")

# ── Console summary table ─────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
cols = ['model', 'slide_acc_TRTR', f'slide_acc_{REGIMEN}', 'delta',
        'ci_diff_lo', 'ci_diff_hi', 'p_bootstrap', 'significant_95',
        'f1_macro_TRTR', f'f1_macro_{REGIMEN}']
print(summary_df[[c for c in cols if c in summary_df.columns]].to_string(index=False))


## 7. Interpretation

In [ ]:
print(f"\n{'='*70}")
print("FINDINGS")
print(f"{'='*70}\n")

model_rows = [r for r in results_all if r['model'] != 'Classification_by_Reconstruction']
sig_models = [r for r in model_rows if r.get('significant_95', False)]
improving  = [r for r in model_rows if r.get('delta', 0) > 0]
degrading  = [r for r in model_rows if r.get('delta', 0) < 0]

print("Differences whose 95% CI excludes 0 (slide-level):")
if sig_models:
    for r in sig_models:
        print(f"  • {r['model']:12s}: delta = {r['delta']:+.4f}  "
              f"95% CI[{r['ci_diff_lo']:+.4f}, {r['ci_diff_hi']:+.4f}]  p = {r['p_bootstrap']:.4f}")
else:
    print("  (None)")

print("\nImprovement trend (CI includes 0):")
for r in improving:
    if not r.get('significant_95', False):
        print(f"  • {r['model']:12s}: delta = {r['delta']:+.4f}  p = {r['p_bootstrap']:.4f}")

if degrading:
    print("\nDegradation:")
    for r in degrading:
        print(f"  • {r['model']:12s}: delta = {r['delta']:+.4f}  p = {r['p_bootstrap']:.4f}")

# CbR-specific findings
print(f"\n{'='*70}")
print("CLASSIFICATION BY RECONSTRUCTION (CbR) - CVAE Signal Matching")
print(f"{'='*70}")

for r in results_all:
    if r["model"] == "Classification_by_Reconstruction":
        print(f"\nCbR Accuracy (baseline):  {r['cbr_accuracy']:.4f}")
        print(f"Bootstrap Mean Accuracy:  {r['cbr_boot_mean']:.4f}")
        print(f"95% Confidence Interval:  [{r['cbr_ci_lo']:.4f}, {r['cbr_ci_hi']:.4f}]")
        print(f"  CI width:        {r['cbr_ci_hi'] - r['cbr_ci_lo']:.4f}")
        print(f"  Slide cohort:    {r['n_slides']} unique slides")
        print(f"  Total samples:   {r['n_samples']} spectra")
        print(f"  Bootstrap iters: {r['n_iterations']}")
        print(f"\nNote: CbR is a reference metric; it is NOT compared statistically with")
        print(f"      TRTR/{REGIMEN}. The CI uses the same slide-level bootstrap methodology.")
